In [48]:
import io
import pandas as pd
import ast
import re
import nltk
from nltk.corpus import stopwords
import numpy as np
from gensim.models import KeyedVectors
import spacy

In [49]:
def process_tweets(file_path):
    """
    Processes the tweets CSV file.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: The processed DataFrame.
    """
    df = pd.read_csv(file_path)
    df['created_at'] = pd.to_datetime(df['created_at']).dt.date
    df.set_index('created_at', inplace=True)
    df.index.name = 'date'
    columns_to_delete = ['Unnamed: 0', 'new_coins', 'user_id']
    df.drop(columns=columns_to_delete, axis=1, inplace=True)
    df.sort_index(inplace=True)

    return df

In [50]:
file_path = 'datasets/tweets.csv'
pre_cleaned_tweets = process_tweets(file_path)
print(pre_cleaned_tweets.head(5))

            favorite_count                                          full_text  \
date                                                                            
2021-02-01             154  #privacy is a human right. learn how to make y...   
2021-02-01              17  overall btc trading volume has increased, but ...   
2021-02-01               3  on average, the return distribution of btc ske...   
2021-02-01            3496  i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...   
2021-02-01               0  rt @reg_mati: la privacidad es un derecho huma...   

           reply_count retweet_count  \
date                                   
2021-02-01          18            23   
2021-02-01           1             5   
2021-02-01           0             1   
2021-02-01         731           686   
2021-02-01           0             7   

                                                   clean_text  \
date                                                            
2021-02-01  privacy h

In [51]:
def merge_tweets_with_market_data(tweets_df, market_data_file_path):
    """
    Merges tweet data with market data on the 'Date' index and saves the merged dataset to a CSV file.

    Parameters:
    market_data_file_path (str): The file path to the market data CSV file.

    Returns:
    pd.DataFrame: The merged DataFrame with 'Date' as the index in datetime.date format.
    """
 
    market_df = pd.read_csv(market_data_file_path, parse_dates=['date'])
    market_df['Date'] = market_df['date'].dt.date
    market_df.set_index('Date', inplace=True)
    merged_df = tweets_df.join(market_df, how='outer')
    merged_df.index.name = 'Date'
    merged_df.sort_index(inplace=True)


    return merged_df

In [52]:
market_data_file_path = 'datasets/BTC_USD_2021-2023.csv'
merged_df = merge_tweets_with_market_data(pre_cleaned_tweets, market_data_file_path)
merged_df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,importance_coefficient,importance_coefficient_normalized,scores,compound,sentiment_type,Unnamed: 14,Unnamed: 15,Unnamed: 16,date,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18,23,privacy human right learn make bitcoin transac...,340,0.000587762440251355,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0,NEUTRAL,NaN,NaN,NaN,2021-02-01,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1,5,overall btc trading volume increased average t...,39.5,6.83E-05,"{'neg': 0.0, 'neu': 0.95, 'pos': 0.05, 'compou...",0.2124,POSITIVE,NaN,NaN,NaN,2021-02-01,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660
2021-02-01,3.0,"on average, the return distribution of btc ske...",0,1,average return distribution btc skews slightly...,7,1.21E-05,"{'neg': 0.053, 'neu': 0.769, 'pos': 0.177, 'co...",0.701,POSITIVE,NaN,NaN,NaN,2021-02-01,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731,686,sent httpstcomfyrz35zjf givedirectly great wor...,8043.5,0.0139049034945934,"{'neg': 0.06, 'neu': 0.856, 'pos': 0.084, 'com...",0.2225,POSITIVE,NaN,NaN,NaN,2021-02-01,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0,7,rt reg_mati la privacidad e un derecho humano ...,7,1.21E-05,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0,NEUTRAL,NaN,NaN,NaN,2021-02-01,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660


In [53]:
def df_drop_cols(df):
    columns_to_drop = ['Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Date', 'date', 'Date.1']
    df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    return df

In [54]:
tweets_btc_df = df_drop_cols(merged_df)
print(tweets_btc_df.head())

            favorite_count                                          full_text  \
Date                                                                            
2021-02-01           154.0  #privacy is a human right. learn how to make y...   
2021-02-01            17.0  overall btc trading volume has increased, but ...   
2021-02-01             3.0  on average, the return distribution of btc ske...   
2021-02-01          3496.0  i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...   
2021-02-01             0.0  rt @reg_mati: la privacidad es un derecho huma...   

           reply_count retweet_count  \
Date                                   
2021-02-01          18            23   
2021-02-01           1             5   
2021-02-01           0             1   
2021-02-01         731           686   
2021-02-01           0             7   

                                                   clean_text  \
Date                                                            
2021-02-01  privacy h

In [55]:
def drop_inconsistent_rows(df, column_name):
    """
    Handles inconsistent rows, some rows were inconsistently shifted after ther merge 
    """
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    df.dropna(subset=[column_name], inplace=True)
    return df

In [56]:
tweets_btc_df_consistent_rows=drop_inconsistent_rows(tweets_btc_df, 'reply_count')
print(tweets_btc_df_consistent_rows)

            favorite_count                                          full_text  \
Date                                                                            
2021-02-01           154.0  #privacy is a human right. learn how to make y...   
2021-02-01            17.0  overall btc trading volume has increased, but ...   
2021-02-01             3.0  on average, the return distribution of btc ske...   
2021-02-01          3496.0  i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...   
2021-02-01             0.0  rt @reg_mati: la privacidad es un derecho huma...   
...                    ...                                                ...   
2023-06-12             3.0  booomð¥\r\n\r\nour #ai bot/indicator crushes...   
2023-06-12             0.0  rt @crypto_crib_: the deadline is today for bi...   
2023-06-12             0.0  rt @crypto_crib_: ð²chinese bank boci issues...   
2023-06-12            56.0  bitcoin, not crypto.\r\n\r\ncrypto, not security.   
2023-06-12             0.0  

In [57]:
def parse_sentiment_scores(df, scores_column):
    df['negative_sentiment_score'] = None
    df['positive_sentiment_score'] = None
    df['neutral_sentiment_score'] = None
    
    for index, row in df.iterrows():
        try:
            if pd.notnull(row[scores_column]) and isinstance(row[scores_column], str):
                score_dict = ast.literal_eval(row[scores_column])
                df.at[index, 'negative_sentiment_score'] = score_dict.get('neg', 0)
                df.at[index, 'positive_sentiment_score'] = score_dict.get('pos', 0)
                df.at[index, 'neutral_sentiment_score'] = score_dict.get('neu', 0)
        except (ValueError, SyntaxError):
            df.at[index, 'negative_sentiment_score'] = pd.NA
            df.at[index, 'positive_sentiment_score'] = pd.NA
            df.at[index, 'neutral_sentiment_score'] = pd.NA
    
    df.dropna(subset=['negative_sentiment_score', 'positive_sentiment_score', 'neutral_sentiment_score'], inplace=True)
    return df


def insert_sentiment_columns_before_open(df):
    """
    Inserts the new sentiment score columns before the 'Open' column.
    """
    open_index = df.columns.get_loc('Open')    
    columns = list(df.columns)
    for col in ['neutral_sentiment_score', 'positive_sentiment_score', 'negative_sentiment_score']:
        columns.insert(open_index, columns.pop(columns.index(col)))
    df = df[columns]
    
    return df

def expand_sentiment_scores(df, scores_column='scores'):
    """
    Expands sentiment scores into individual columns, inserts them before the 'Open' column,
    and saves the updated DataFrame to a new CSV file.
    """
    df = parse_sentiment_scores(df, scores_column)  
    df = insert_sentiment_columns_before_open(df)    
    df.drop(columns=[scores_column], inplace=True)
    return df

In [58]:
tweets_btc_df_clean_sent_score=expand_sentiment_scores(tweets_btc_df_consistent_rows, 'scores')
print(tweets_btc_df_clean_sent_score.head())

            favorite_count                                          full_text  \
Date                                                                            
2021-02-01           154.0  #privacy is a human right. learn how to make y...   
2021-02-01            17.0  overall btc trading volume has increased, but ...   
2021-02-01             3.0  on average, the return distribution of btc ske...   
2021-02-01          3496.0  i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...   
2021-02-01             0.0  rt @reg_mati: la privacidad es un derecho huma...   

            reply_count retweet_count  \
Date                                    
2021-02-01         18.0            23   
2021-02-01          1.0             5   
2021-02-01          0.0             1   
2021-02-01        731.0           686   
2021-02-01          0.0             7   

                                                   clean_text  \
Date                                                            
2021-02-01  pr

C:\Users\natha\AppData\Local\Temp\ipykernel_3636\2529300530.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=[scores_column], inplace=True)


In [59]:
def calculate_historical_volatility(df):
    df['daily_return'] = (df['Close'] - df['Open']) / df['Open']
    df['historical_volatility'] = df['daily_return'].expanding(min_periods=1).std()
    return df

In [60]:
tweets_btc_with_volatility = calculate_historical_volatility(tweets_btc_df_clean_sent_score)
print(tweets_btc_with_volatility.head())

            favorite_count                                          full_text  \
Date                                                                            
2021-02-01           154.0  #privacy is a human right. learn how to make y...   
2021-02-01            17.0  overall btc trading volume has increased, but ...   
2021-02-01             3.0  on average, the return distribution of btc ske...   
2021-02-01          3496.0  i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...   
2021-02-01             0.0  rt @reg_mati: la privacidad es un derecho huma...   

            reply_count retweet_count  \
Date                                    
2021-02-01         18.0            23   
2021-02-01          1.0             5   
2021-02-01          0.0             1   
2021-02-01        731.0           686   
2021-02-01          0.0             7   

                                                   clean_text  \
Date                                                            
2021-02-01  pr

In [61]:
def calculate_price_derivatives(df):
    df['first_derivative'] = df['Close'].diff(1)
    df['second_derivative'] = df['first_derivative'].diff(1)
    return df

In [62]:
tweets_btc_with_derivatives = calculate_price_derivatives(tweets_btc_with_volatility)
tweets_btc_with_derivatives.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,Open,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18.0,23,privacy human right learn make bitcoin transac...,340,0.000587762440251355,0,NEUTRAL,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,NaN,NaN,NaN
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1.0,5,overall btc trading volume increased average t...,39.5,6.83E-05,0.2124,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,NaN
2021-02-01,3.0,"on average, the return distribution of btc ske...",0.0,1,average return distribution btc skews slightly...,7,1.21E-05,0.701,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent httpstcomfyrz35zjf givedirectly great wor...,8043.5,0.0139049034945934,0.2225,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0.0,7,rt reg_mati la privacidad e un derecho humano ...,7,1.21E-05,0,NEUTRAL,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0


In [63]:
tweets_btc_with_derivatives.to_csv('datasets/Pre_cleaned_dataset.csv')

In [64]:
# Download the set of stopwords from nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\natha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [65]:
def clean_individual_text(text):
    text = re.sub(r'http\S+', '', text) # no links
    text = re.sub(r'[^a-zA-Z\s]', '', text) # no symbols, no punct
    text = text.lower()
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_column(dataframe, column_name):
    dataframe[column_name] = dataframe[column_name].apply(clean_individual_text)
    return dataframe

In [66]:
cleaned_df = clean_text_column(tweets_btc_with_derivatives, 'clean_text')
cleaned_df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,Open,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18.0,23,privacy human right learn make bitcoin transac...,340,0.000587762440251355,0,NEUTRAL,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,NaN,NaN,NaN
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1.0,5,overall btc trading volume increased average t...,39.5,6.83E-05,0.2124,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,NaN
2021-02-01,3.0,"on average, the return distribution of btc ske...",0.0,1,average return distribution btc skews slightly...,7,1.21E-05,0.701,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,8043.5,0.0139049034945934,0.2225,POSITIVE,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0.0,7,rt regmati la privacidad e un derecho humano m...,7,1.21E-05,0,NEUTRAL,0.0,...,33114.578125,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0


In [67]:
cleaned_df.to_csv('datasets/Pre_cleaned_dataset.csv')

In [68]:
def target_variable(dataframe, open_column, close_column, target_column_name='target'):
    """
    This function creates a target variable column in the dataframe indicating whether the price of Bitcoin went up or not.
    
    :param dataframe: A pandas DataFrame containing the open and close prices
    :param open_column: The name of the column containing the opening prices
    :param close_column: The name of the column containing the closing prices
    :param target_column_name: The name of the new target variable column to be added to the dataframe
    :return: A pandas DataFrame with the new target variable column added
    """
    dataframe[target_column_name] = (dataframe[close_column] > dataframe[open_column]).astype(int)
    return dataframe


In [69]:
# Create the target variable and add it to the DataFrame
df = target_variable(cleaned_df, 'Open', 'Close')
df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18.0,23,privacy human right learn make bitcoin transac...,340,0.000587762440251355,0,NEUTRAL,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,NaN,NaN,NaN,1
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1.0,5,overall btc trading volume increased average t...,39.5,6.83E-05,0.2124,POSITIVE,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,NaN,1
2021-02-01,3.0,"on average, the return distribution of btc ske...",0.0,1,average return distribution btc skews slightly...,7,1.21E-05,0.701,POSITIVE,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,8043.5,0.0139049034945934,0.2225,POSITIVE,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0.0,7,rt regmati la privacidad e un derecho humano m...,7,1.21E-05,0,NEUTRAL,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1


In [70]:
df.to_csv('datasets/Pre_cleaned_dataset.csv')

In [71]:
def tweet_to_vector(tweet, model):
    words = tweet.split()
    word_vectors = [model[word] for word in words if word in model]
    if not word_vectors:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

In [72]:
# Load pre-trained Word2Vec 
word2vec_path = 'GoogleNews-vectors-negative300.bin'
word2vec = KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

df['clean_text_vector'] = df['clean_text'].apply(lambda x: tweet_to_vector(x, word2vec))


In [73]:
df.to_csv('datasets/Pre_cleaned_dataset.csv')

In [74]:
df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target,clean_text_vector
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18.0,23,privacy human right learn make bitcoin transac...,340,0.000587762440251355,0,NEUTRAL,0.0,...,32384.228516,33537.175781,33537.175781,61400400660,0.012762,NaN,NaN,NaN,1,"[-0.0019989014, 0.06149292, -0.007598877, 0.09..."
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1.0,5,overall btc trading volume increased average t...,39.5,6.83E-05,0.2124,POSITIVE,0.0,...,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,NaN,1,"[7.8986675e-05, 0.07419362, -0.057335347, 0.05..."
2021-02-01,3.0,"on average, the return distribution of btc ske...",0.0,1,average return distribution btc skews slightly...,7,1.21E-05,0.701,POSITIVE,0.0,...,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1,"[0.033384148, 0.010453657, -0.04193826, 0.0561..."
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,8043.5,0.0139049034945934,0.2225,POSITIVE,0.0,...,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1,"[0.06558048, -0.040882785, 0.05294261, 0.11822..."
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0.0,7,rt regmati la privacidad e un derecho humano m...,7,1.21E-05,0,NEUTRAL,0.0,...,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1,"[-0.03623693, 0.041989494, 0.12695672, 0.12555..."


In [75]:
def reposition_column(df, column_to_move, reference_column):
    cols = df.columns.tolist()
    ref_index = cols.index(reference_column)
    cols.remove(column_to_move)
    cols.insert(ref_index, column_to_move)
    
    return df.reindex(columns=cols)

In [76]:
df = reposition_column(df, 'clean_text_vector', 'importance_coefficient')

In [77]:
def convert_and_filter_sentiment(df, column_name):
    df = df[df[column_name] != 0]
    sentiment_mapping = {'POSITIVE': 1, 'NEUTRAL': 0, 'NEGATIVE': -1}
    df[column_name] = df[column_name].map(sentiment_mapping)
    df = df.dropna(subset=[column_name])
    df[column_name] = df[column_name].astype(int)
    return df

In [78]:
df = convert_and_filter_sentiment(df, 'sentiment_type')
df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,154.0,#privacy is a human right. learn how to make y...,18.0,23,privacy human right learn make bitcoin transac...,"[-0.0019989014, 0.06149292, -0.007598877, 0.09...",340,0.000587762440251355,0,0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,NaN,NaN,NaN,1
2021-02-01,17.0,"overall btc trading volume has increased, but ...",1.0,5,overall btc trading volume increased average t...,"[7.8986675e-05, 0.07419362, -0.057335347, 0.05...",39.5,6.83E-05,0.2124,1,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,NaN,1
2021-02-01,3.0,"on average, the return distribution of btc ske...",0.0,1,average return distribution btc skews slightly...,"[0.033384148, 0.010453657, -0.04193826, 0.0561...",7,1.21E-05,0.701,1,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1
2021-02-01,0.0,rt @reg_mati: la privacidad es un derecho huma...,0.0,7,rt regmati la privacidad e un derecho humano m...,"[-0.03623693, 0.041989494, 0.12695672, 0.12555...",7,1.21E-05,0,0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.0,0.0,0.0,1


In [79]:
df.to_csv('datasets/Pre_cleaned_dataset.csv')


In [80]:
def select_highest_favorites(df, fav_col='favorite_count'):
    df_sorted = df.sort_values(by=fav_col, ascending=False)
    df_highest_favorites = df_sorted.groupby(df_sorted.index).first()
    return df_highest_favorites

In [81]:
df = select_highest_favorites(df)
df.head()

,favorite_count,full_text,reply_count,retweet_count,clean_text,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.000000,0.000000,0.000000,1
2021-02-02,109.0,watch this video to learn the truth about what...,5.0,25,watch video learn truth doublespend separate f...,"[0.041521344, -0.015838623, -0.019496372, 0.05...",245.5,0.000424399056122669,0.0516,1,...,35896.882813,33489.218750,35510.289063,35510.289063,63088585433,0.058959,0.015399,1973.113282,1973.113282,1
2021-02-03,153.0,"in this 7min video, i review strategies from s...",9.0,24,min video review strategy simple advanced help...,"[-0.015966797, 0.065028384, -0.05678711, 0.042...",334.5,0.00057825451842376,0.5719,1,...,37480.187500,35443.984375,37472.089844,37472.089844,61166818159,0.055230,0.022934,0.000000,0.000000,1
2021-02-04,196.0,to be fair elon gave all of you the chance to ...,31.0,18,fair elon gave chance load bag early,"[0.090413414, 0.091308594, 0.001200358, 0.0492...",425.5,0.000735567406843975,0.5106,1,...,38592.175781,36317.500000,36926.066406,36926.066406,68838074392,-0.014651,0.030463,0.000000,0.000000,0
2021-02-05,100.0,â30 minutes into the video and i've learned ...,7.0,8,minute video ive learned much cant smash like ...,"[0.024189342, 0.055383857, -0.016594626, 0.096...",211.5,0.000365622812097534,0.8301,1,...,38225.906250,36658.761719,38144.308594,38144.308594,58598066402,0.032838,0.025996,0.000000,0.000000,1


In [82]:
df = calculate_price_derivatives(df)
df

,favorite_count,full_text,reply_count,retweet_count,clean_text,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-01,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,sent givedirectly great work distributing fund...,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.000000,NaN,NaN,1
2021-02-02,109.0,watch this video to learn the truth about what...,5.0,25,watch video learn truth doublespend separate f...,"[0.041521344, -0.015838623, -0.019496372, 0.05...",245.5,0.000424399056122669,0.0516,1,...,35896.882813,33489.218750,35510.289063,35510.289063,63088585433,0.058959,0.015399,1973.113282,NaN,1
2021-02-03,153.0,"in this 7min video, i review strategies from s...",9.0,24,min video review strategy simple advanced help...,"[-0.015966797, 0.065028384, -0.05678711, 0.042...",334.5,0.00057825451842376,0.5719,1,...,37480.187500,35443.984375,37472.089844,37472.089844,61166818159,0.055230,0.022934,1961.800781,-11.312501,1
2021-02-04,196.0,to be fair elon gave all of you the chance to ...,31.0,18,fair elon gave chance load bag early,"[0.090413414, 0.091308594, 0.001200358, 0.0492...",425.5,0.000735567406843975,0.5106,1,...,38592.175781,36317.500000,36926.066406,36926.066406,68838074392,-0.014651,0.030463,-546.023438,-2507.824219,0
2021-02-05,100.0,â30 minutes into the video and i've learned ...,7.0,8,minute video ive learned much cant smash like ...,"[0.024189342, 0.055383857, -0.016594626, 0.096...",211.5,0.000365622812097534,0.8301,1,...,38225.906250,36658.761719,38144.308594,38144.308594,58598066402,0.032838,0.025996,1218.242188,1764.265626,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-06-08,21045.0,this is simply false. not sure if it's the jou...,2927.0,4199,simply false sure journalist source best knowl...,"[0.07864889, -0.06778717, 0.0038274128, 0.0821...",47752.5,0.082550370376773,0.5593,1,...,26797.513672,26246.664063,26508.216797,26508.216797,11904824295,0.006094,0.032423,162.218750,1055.003906,1
2023-06-09,10739.0,"just in: robinhood to delist cardano, polygon,...",1338.0,2355,robinhood delist cardano polygon solana,"[-0.046875, -0.0803833, 0.1640625, 0.060424805...",24502,0.0423569273854079,0,0,...,26770.289063,26339.314453,26480.375000,26480.375000,11015551640,-0.000964,0.032325,-27.841797,-190.060547,0
2023-06-10,13594.0,"according to our data, last 24hrs, @binance ne...",2138.0,2460,according data last hrs binance net outflow wa...,"[-0.036156874, -0.027393047, -0.009594257, 0.0...",30717,0.0531008790505908,0.4019,1,...,26531.044922,25501.835938,25851.240234,25851.240234,19872933189,-0.023810,0.032155,-629.134766,-601.292969,0


In [83]:
def create_multiindex(df, text_col):
    if text_col not in df.columns:
        raise KeyError(f"{text_col} is not a column in the DataFrame")
    df.set_index([df.index, text_col], inplace=True)
    return df

In [84]:
df = create_multiindex(df, 'clean_text')
df.head()

,,favorite_count,full_text,reply_count,retweet_count,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,clean_text,,,,,,,,,,,,,,,,,,,,,
2021-02-01,sent givedirectly great work distributing fund directly world poorest take crypto dont yet take doge directly maybe change,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,0.0,...,34638.214844,32384.228516,33537.175781,33537.175781,61400400660,0.012762,0.000000,NaN,NaN,1
2021-02-02,watch video learn truth doublespend separate fud fact bitcoin,109.0,watch this video to learn the truth about what...,5.0,25,"[0.041521344, -0.015838623, -0.019496372, 0.05...",245.5,0.000424399056122669,0.0516,1,0.0,...,35896.882813,33489.218750,35510.289063,35510.289063,63088585433,0.058959,0.015399,1973.113282,NaN,1
2021-02-03,min video review strategy simple advanced help maintain privacy using bitcoin,153.0,"in this 7min video, i review strategies from s...",9.0,24,"[-0.015966797, 0.065028384, -0.05678711, 0.042...",334.5,0.00057825451842376,0.5719,1,0.0,...,37480.187500,35443.984375,37472.089844,37472.089844,61166818159,0.055230,0.022934,1961.800781,-11.312501,1
2021-02-04,fair elon gave chance load bag early,196.0,to be fair elon gave all of you the chance to ...,31.0,18,"[0.090413414, 0.091308594, 0.001200358, 0.0492...",425.5,0.000735567406843975,0.5106,1,0.098,...,38592.175781,36317.500000,36926.066406,36926.066406,68838074392,-0.014651,0.030463,-546.023438,-2507.824219,0
2021-02-05,minute video ive learned much cant smash like button hard enough one top comment new video bitcoin double spending wow im humbled honored,100.0,â30 minutes into the video and i've learned ...,7.0,8,"[0.024189342, 0.055383857, -0.016594626, 0.096...",211.5,0.000365622812097534,0.8301,1,0.075,...,38225.906250,36658.761719,38144.308594,38144.308594,58598066402,0.032838,0.025996,1218.242188,1764.265626,1


In [85]:
def fillna_with_mean_and_round(df):
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].fillna(df[column].mean())
            df[column] = df[column].round(2)
    return df

In [86]:
df = fillna_with_mean_and_round(df)
df.head()

,,favorite_count,full_text,reply_count,retweet_count,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,High,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target
Date,clean_text,,,,,,,,,,,,,,,,,,,,,
2021-02-01,sent givedirectly great work distributing fund directly world poorest take crypto dont yet take doge directly maybe change,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,0.0,...,34638.21,32384.23,33537.18,33537.18,61400400660,0.01,0.00,-9.08,-2.39,1
2021-02-02,watch video learn truth doublespend separate fud fact bitcoin,109.0,watch this video to learn the truth about what...,5.0,25,"[0.041521344, -0.015838623, -0.019496372, 0.05...",245.5,0.000424399056122669,0.0516,1,0.0,...,35896.88,33489.22,35510.29,35510.29,63088585433,0.06,0.02,1973.11,-2.39,1
2021-02-03,min video review strategy simple advanced help maintain privacy using bitcoin,153.0,"in this 7min video, i review strategies from s...",9.0,24,"[-0.015966797, 0.065028384, -0.05678711, 0.042...",334.5,0.00057825451842376,0.5719,1,0.0,...,37480.19,35443.98,37472.09,37472.09,61166818159,0.06,0.02,1961.80,-11.31,1
2021-02-04,fair elon gave chance load bag early,196.0,to be fair elon gave all of you the chance to ...,31.0,18,"[0.090413414, 0.091308594, 0.001200358, 0.0492...",425.5,0.000735567406843975,0.5106,1,0.098,...,38592.18,36317.50,36926.07,36926.07,68838074392,-0.01,0.03,-546.02,-2507.82,0
2021-02-05,minute video ive learned much cant smash like button hard enough one top comment new video bitcoin double spending wow im humbled honored,100.0,â30 minutes into the video and i've learned ...,7.0,8,"[0.024189342, 0.055383857, -0.016594626, 0.096...",211.5,0.000365622812097534,0.8301,1,0.075,...,38225.91,36658.76,38144.31,38144.31,58598066402,0.03,0.03,1218.24,1764.27,1


In [87]:
df.to_csv('datasets/Preprocessed_csv.csv')

In [88]:
nlp = spacy.load('en_core_web_sm')

#### __*If OSError:*__
[E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

#### __*Solution :*__ 
In your terminal run the following command
```shell
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.0/en_core_web_sm-3.7.0-py3-none-any.whl#sha256=6215d71a3212690e9aec49408a27e3fe6ad7cd6c715476e93d70dc784041e93e
```

In [89]:
word2vec_path = 'GoogleNews-vectors-negative300.bin'
word2vec = KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

In [90]:
def extract_and_vectorize_entities(df):
    def get_entities(text):
        return [ent.text for ent in nlp(text).ents]

    def entities_to_vector(entities):
        entity_vectors = [word2vec[entity] for entity in entities if entity in word2vec]
        if not entity_vectors:
            return np.zeros(word2vec.vector_size)
        return np.mean(entity_vectors, axis=0)

    df['entity_vectors'] = df.index.get_level_values('clean_text').map(lambda x: entities_to_vector(get_entities(x)))

    return df

In [91]:
extracted_vectorized_df = extract_and_vectorize_entities(df)
extracted_vectorized_df.head(5)

,,favorite_count,full_text,reply_count,retweet_count,clean_text_vector,importance_coefficient,importance_coefficient_normalized,compound,sentiment_type,negative_sentiment_score,...,Low,Close,Adj Close,Volume,daily_return,historical_volatility,first_derivative,second_derivative,target,entity_vectors
Date,clean_text,,,,,,,,,,,,,,,,,,,,,
2021-02-01,sent givedirectly great work distributing fund directly world poorest take crypto dont yet take doge directly maybe change,3496.0,i sent some! https://t.co/mfyrz35zjf\r\n\r\nyo...,731.0,686,"[0.06558048, -0.040882785, 0.05294261, 0.11822...",8043.5,0.0139049034945934,0.2225,1,0.0,...,32384.23,33537.18,33537.18,61400400660,0.01,0.00,-9.08,-2.39,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2021-02-02,watch video learn truth doublespend separate fud fact bitcoin,109.0,watch this video to learn the truth about what...,5.0,25,"[0.041521344, -0.015838623, -0.019496372, 0.05...",245.5,0.000424399056122669,0.0516,1,0.0,...,33489.22,35510.29,35510.29,63088585433,0.06,0.02,1973.11,-2.39,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2021-02-03,min video review strategy simple advanced help maintain privacy using bitcoin,153.0,"in this 7min video, i review strategies from s...",9.0,24,"[-0.015966797, 0.065028384, -0.05678711, 0.042...",334.5,0.00057825451842376,0.5719,1,0.0,...,35443.98,37472.09,37472.09,61166818159,0.06,0.02,1961.80,-11.31,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2021-02-04,fair elon gave chance load bag early,196.0,to be fair elon gave all of you the chance to ...,31.0,18,"[0.090413414, 0.091308594, 0.001200358, 0.0492...",425.5,0.000735567406843975,0.5106,1,0.098,...,36317.50,36926.07,36926.07,68838074392,-0.01,0.03,-546.02,-2507.82,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2021-02-05,minute video ive learned much cant smash like button hard enough one top comment new video bitcoin double spending wow im humbled honored,100.0,â30 minutes into the video and i've learned ...,7.0,8,"[0.024189342, 0.055383857, -0.016594626, 0.096...",211.5,0.000365622812097534,0.8301,1,0.075,...,36658.76,38144.31,38144.31,58598066402,0.03,0.03,1218.24,1764.27,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [92]:
df = reposition_column(df, 'entity_vectors', 'clean_text_vector')

In [93]:
df.to_csv('datasets/Preprocessed_csv.csv')

In [ ]:
df.drop('entity_vectors', axis=1, inplace=True)

In [ ]:
!pip freeze > requirements.txt